# 02 — Emotion Classifier (DistilBERT)

### Setup steps
1. Go to [kaggle.com/code](https://www.kaggle.com/code) → **+ New Notebook**
2. Settings → Accelerator → **GPU T4 x2** (free tier)
3. Paste or upload this notebook
4. **Run All** — training takes ~15-20 minutes
5. Download `/kaggle/working/emotion_classifier_trained.zip`
6. Unzip into your local `models/emotion_classifier/`

### What this does
- Downloads `dair-ai/emotion` directly from HuggingFace
- Fine-tunes `distilbert-base-uncased` for 5 epochs with early stopping
- Uses **weighted cross-entropy** to handle 10:1 class imbalance (surprise vs sadness)
- Evaluates on test set and saves the best checkpoint

In [ ]:
# ── Install / upgrade packages ────────────────────────────────────────────
!pip install -q datasets transformers[torch] accelerate scikit-learn seaborn

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import json, time, os, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128
BATCH_SIZE = 32          # 32 fits comfortably on T4 16 GB
LR         = 2e-5
EPOCHS     = 5
PATIENCE   = 2           # early stopping patience
OUT_DIR    = Path('/kaggle/working/emotion_classifier')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Label maps
EMOTION_LABELS = ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']
LABEL2ID = {e: i for i, e in enumerate(EMOTION_LABELS)}
ID2LABEL = {i: e for i, e in enumerate(EMOTION_LABELS)}
# dair-ai/emotion native int labels
HF_ID2LABEL = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
print('Config ready')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
print('Loading dair-ai/emotion …')
ds = load_dataset('dair-ai/emotion')

def hf_to_df(split):
    df = ds[split].to_pandas()
    df['emotion'] = df['label'].map(HF_ID2LABEL)
    df = df.drop(columns=['label'])
    return df

train_df = hf_to_df('train')
val_df   = hf_to_df('validation')
test_df  = hf_to_df('test')

print(f'  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print('\nClass distribution (train):')
for emo, cnt in train_df['emotion'].value_counts().items():
    pct = cnt/len(train_df)*100
    bar = '█' * int(pct/2)
    print(f'  {emo:<10} {cnt:>5}  {pct:>5.1f}%  {bar}')

vc = train_df['emotion'].value_counts()
print(f'\nImbalance ratio: {vc.max()/vc.min():.1f}x  → using weighted cross-entropy')

In [ ]:
# ── EDA: sample texts per emotion ────────────────────────────────────────
print('Sample texts per emotion class:')
print('-' * 65)
for emo in EMOTION_LABELS:
    sample = train_df[train_df['emotion'] == emo]['text'].iloc[0]
    print(f'  [{emo:<8}]  {sample[:60]}')

In [ ]:
# ── EDA: text length distribution ─────────────────────────────────────────
train_df['word_count'] = train_df['text'].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_df['word_count'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Word count distribution'); axes[0].set_xlabel('Words')

counts = train_df['emotion'].value_counts()
colors = ['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6','#1abc9c']
axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[1].set_title('Class distribution'); axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('Emotion Dataset — EDA', fontsize=13)
plt.tight_layout(); plt.show()
print(train_df['word_count'].describe().round(1).to_dict())

In [ ]:
# ── Dataset & DataLoader ───────────────────────────────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(
            [LABEL2ID[l] for l in labels], dtype=torch.long
        )
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i):
        return {
            'input_ids':      self.enc['input_ids'][i],
            'attention_mask': self.enc['attention_mask'][i],
            'labels':         self.labels[i],
        }

def make_loader(df, shuffle=False):
    ds_ = EmotionDataset(df['text'].tolist(), df['emotion'].tolist(), tokenizer, MAX_LEN)
    return DataLoader(ds_, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df)
test_loader  = make_loader(test_df)
print(f'Loaders ready  train={len(train_loader)} batches  val={len(val_loader)}  test={len(test_loader)}')

In [ ]:
# ── Model + weighted loss ─────────────────────────────────────────────────
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(EMOTION_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(DEVICE)

# Class weights — weighted cross-entropy for 10:1 imbalance
label_ids = np.array([LABEL2ID[l] for l in train_df['emotion']])
class_wts = compute_class_weight(
    'balanced',
    classes=np.arange(len(EMOTION_LABELS)),
    y=label_ids,
)
weights_tensor = torch.tensor(class_wts, dtype=torch.float).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=weights_tensor)

print('Class weights:')
for emo, w in zip(EMOTION_LABELS, class_wts):
    print(f'  {emo:<10} {w:.3f}')

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler = torch.amp.GradScaler()
print(f'\nOptimiser ready  total_steps={total_steps}  warmup={warmup_steps}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total = 0.0
    for batch in loader:
        optimizer.zero_grad()
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        with torch.amp.autocast('cuda'):
            loss = loss_fn(model(ids, attention_mask=mask).logits, lbl)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, loss_fn):
    model.eval()
    total, correct, n = 0.0, 0, 0
    all_pred, all_true = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        logits = model(ids, attention_mask=mask).logits
        total += loss_fn(logits, lbl).item()
        pred = logits.argmax(-1)
        correct += (pred == lbl).sum().item(); n += lbl.size(0)
        all_pred.extend(pred.cpu().numpy()); all_true.extend(lbl.cpu().numpy())
    return total/len(loader), correct/n, np.array(all_pred), np.array(all_true)

best_f1, patience_left = -1.0, PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    t0         = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
    val_loss, val_acc, val_pred, val_true = eval_epoch(model, val_loader, loss_fn)
    val_f1     = f1_score(val_true, val_pred, average='macro')
    elapsed    = time.time() - t0

    row = dict(epoch=epoch, train_loss=round(train_loss,4),
               val_loss=round(val_loss,4), val_acc=round(val_acc,4),
               val_f1=round(val_f1,4), elapsed=round(elapsed,1))
    history.append(row)

    marker = ''
    if val_f1 > best_f1:
        best_f1 = val_f1; patience_left = PATIENCE
        model.save_pretrained(str(OUT_DIR))
        tokenizer.save_pretrained(str(OUT_DIR))
        marker = '  ← best ✓'
    else:
        patience_left -= 1

    print(f'Epoch {epoch}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'acc={val_acc:.4f}  F1={val_f1:.4f}  ({elapsed:.0f}s){marker}')

    if patience_left <= 0:
        print(f'\nEarly stopping after epoch {epoch}')
        break

print(f'\nBest val macro-F1: {best_f1:.4f}')

In [ ]:
# ── Training history plot ─────────────────────────────────────────────────
epochs_     = [r['epoch']      for r in history]
train_losses= [r['train_loss'] for r in history]
val_losses  = [r['val_loss']   for r in history]
val_accs    = [r['val_acc']    for r in history]
val_f1s     = [r['val_f1']     for r in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs_, train_losses, 'o-', label='train', color='#3498db')
axes[0].plot(epochs_, val_losses,   'o-', label='val',   color='#e74c3c')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs_, val_accs, 'o-', color='#2ecc71')
axes[1].set_title('Val Accuracy'); axes[1].set_ylim(0,1)
axes[2].plot(epochs_, val_f1s, 'o-', color='#9b59b6')
axes[2].set_title('Val Macro-F1'); axes[2].set_ylim(0,1)
plt.suptitle('DistilBERT Training History', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── Test evaluation ───────────────────────────────────────────────────────
# Reload best model
best_model = DistilBertForSequenceClassification.from_pretrained(str(OUT_DIR)).to(DEVICE)

test_loss, test_acc, test_pred, test_true = eval_epoch(best_model, test_loader, loss_fn)
test_f1 = f1_score(test_true, test_pred, average='macro')

pred_labels = [ID2LABEL[i] for i in test_pred]
true_labels = [ID2LABEL[i] for i in test_true]

print(f'Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Test macro-F1 : {test_f1:.4f}')
print()
print(classification_report(true_labels, pred_labels, target_names=EMOTION_LABELS))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(true_labels, pred_labels, labels=EMOTION_LABELS)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
            linewidths=0.4, ax=ax)
ax.set_title(f'Confusion Matrix  (acc={test_acc:.4f})', fontsize=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

# Hardest emotions
cr = classification_report(true_labels, pred_labels, output_dict=True)
sorted_emo = sorted(EMOTION_LABELS, key=lambda e: cr[e]['f1-score'])
print('Hardest emotions:')
for emo in sorted_emo[:3]:
    print(f'  {emo:<10} F1={cr[emo]["f1-score"]:.3f}')

In [ ]:
# ── Save all artefacts ────────────────────────────────────────────────────

# Training log
with open(OUT_DIR / 'training_log.txt', 'w') as f:
    f.write('epoch,train_loss,val_loss,val_acc,val_f1,elapsed_s\n')
    for r in history:
        f.write(f"{r['epoch']},{r['train_loss']},{r['val_loss']},{r['val_acc']},{r['val_f1']},{r['elapsed']}\n")

# Train config
train_cfg = dict(
    model_name=MODEL_NAME, num_labels=len(EMOTION_LABELS),
    label2id=LABEL2ID, id2label={str(k): v for k,v in ID2LABEL.items()},
    lr=LR, batch_size=BATCH_SIZE, max_length=MAX_LEN,
    patience=PATIENCE, imbalance_strategy='weighted_cross_entropy',
    best_val_f1=best_f1, test_accuracy=round(test_acc, 4),
    test_macro_f1=round(test_f1, 4),
)
with open(OUT_DIR / 'train_config.json', 'w') as f:
    json.dump(train_cfg, f, indent=2)

# Zip for download
zip_path = '/kaggle/working/emotion_classifier_trained.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in OUT_DIR.rglob('*'):
        if fpath.is_file():
            zf.write(fpath, fpath.relative_to(OUT_DIR))

size_mb = Path(zip_path).stat().st_size / 1e6
print(f'\n✅ Saved to {zip_path}  ({size_mb:.1f} MB)')
print('\nFiles in OUT_DIR:')
for f in sorted(OUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:<40} {f.stat().st_size/1e6:.2f} MB')

In [ ]:
# ── Quick inference test on Kaggle ────────────────────────────────────────
from transformers import pipeline

pipe = pipeline(
    'text-classification',
    model=str(OUT_DIR),
    tokenizer=str(OUT_DIR),
    device=0 if DEVICE.type == 'cuda' else -1,
    top_k=None,
)

test_cases = [
    "I feel absolutely hopeless and want to give up everything",
    "This is the best day of my life, I am so happy!",
    "I am furious about what just happened, this is unacceptable",
    "I am so scared and nervous, I cannot stop shaking",
    "I love you so much, you mean everything to me",
    "Wow I never expected that, completely shocked!",
]

print(f'  {"Text":<50}  {"Emotion":<10}  {"Conf":>6}')
print('-' * 72)
for text in test_cases:
    t0  = time.time()
    out = pipe(text)
    ms  = (time.time() - t0) * 1000
    top = max(out[0], key=lambda x: x['score'])
    print(f"  {text[:48]:<50}  {top['label']:<10}  {top['score']:>6.3f}  ({ms:.0f}ms)")

print('\n✅ All done! Download emotion_classifier_trained.zip from the Output panel.')